In [ ]:
# NHIỆM VỤ 3: PHÂN TÍCH VÀ TỔNG HỢP DỮ LIỆU ĐIỂM DANH

import pandas as pd

print("PHÂN TÍCH SINH VIÊN")

student_summary = df_final.groupby(['student_id', 'full_name', 'class_cohort']).agg({
    'date': 'count',
    'status': lambda x: (x == 'Có mặt').sum(),
}).rename(columns={'date': 'total_sessions', 'status': 'present_count'})

absent_count = df_final[df_final['status'] == 'Vắng'].groupby('student_id').size()
student_summary['absent_count'] = absent_count

late_count = df_final[df_final['status'] == 'Đi muộn'].groupby('student_id').size()
student_summary['late_count'] = late_count

student_summary = student_summary.fillna(0)
student_summary['absent_count'] = student_summary['absent_count'].astype(int)
student_summary['late_count'] = student_summary['late_count'].astype(int)

student_summary['attendance_rate'] = (
    student_summary['present_count'] / student_summary['total_sessions'] * 100
).round(2)

def classify_student(rate):
    if rate >= 90:
        return 'Xuất sắc'
    elif rate >= 75:
        return 'Tốt'
    elif rate >= 60:
        return 'Trung bình'
    else:
        return 'Yếu'

student_summary['classification'] = student_summary['attendance_rate'].apply(classify_student)

student_summary = student_summary.sort_values('attendance_rate', ascending=False)

print("BẢNG TỔNG HỢP THEO SINH VIÊN:")
print(student_summary.to_string())

print("THỐNG KÊ CHUNG:")
print(f"  • Tổng số sinh viên: {len(student_summary)}")
print(f"  • Tỷ lệ điểm danh trung bình: {student_summary['attendance_rate'].mean():.2f}%")
print(f"  • Tỷ lệ cao nhất: {student_summary['attendance_rate'].max():.2f}%")
print(f"  • Tỷ lệ thấp nhất: {student_summary['attendance_rate'].min():.2f}%")

print("PHÂN LOẠI SINH VIÊN:")
classification_counts = student_summary['classification'].value_counts()
for classification, count in classification_counts.items():
    print(f"  • {classification}: {count} sinh viên")


print("PHÂN TÍCH THEO LỚP HỌC")

class_summary = df_final.groupby(['class_id', 'course_name']).agg({
    'student_id': 'nunique',
    'date': 'count',
}).rename(columns={'student_id': 'num_students', 'date': 'total_records'})

class_status = df_final.groupby(['class_id', 'status']).size().unstack(fill_value=0)
class_summary = class_summary.join(class_status)

if 'Có mặt' in class_summary.columns:
    class_summary['avg_attendance_rate'] = (
        class_summary['Có mặt'] / class_summary['total_records'] * 100
    ).round(2)
else:
    class_summary['avg_attendance_rate'] = 0

class_summary['avg_sessions_per_student'] = (
    class_summary['total_records'] / class_summary['num_students']
).round(2)

print("BẢNG TỔNG HỢP THEO LỚP:")
print(class_summary.to_string())

print("THỐNG KÊ THEO LỚP:")
print(f"  • Tổng số lớp: {len(class_summary)}")
print(f"  • Số sinh viên TB/lớp: {class_summary['num_students'].mean():.2f}")
print(f"  • Tỷ lệ điểm danh cao nhất: {class_summary['avg_attendance_rate'].max():.2f}%")
print(f"  • Tỷ lệ điểm danh thấp nhất: {class_summary['avg_attendance_rate'].min():.2f}%")

print("PHÂN TÍCH THEO KHÓA HỌC")

cohort_summary = df_final.groupby('class_cohort').agg({
    'student_id': 'nunique',
    'date': 'count',
}).rename(columns={'student_id': 'num_students', 'date': 'total_records'})

cohort_status = df_final.groupby(['class_cohort', 'status']).size().unstack(fill_value=0)
cohort_summary = cohort_summary.join(cohort_status)

if 'Có mặt' in cohort_summary.columns:
    cohort_summary['attendance_rate'] = (
        cohort_summary['Có mặt'] / cohort_summary['total_records'] * 100
    ).round(2)
else:
    cohort_summary['attendance_rate'] = 0


if 'Vắng' in cohort_summary.columns:
    cohort_summary['absent_rate'] = (
        cohort_summary['Vắng'] / cohort_summary['total_records'] * 100
    ).round(2)
else:
    cohort_summary['absent_rate'] = 0


cohort_summary = cohort_summary.sort_values('attendance_rate', ascending=False)

print("BẢNG TỔNG HỢP THEO KHÓA:")
print(cohort_summary.to_string())

print("SO SÁNH GIỮA CÁC KHÓA:")
for cohort in cohort_summary.index:
    students = cohort_summary.loc[cohort, 'num_students']
    rate = cohort_summary.loc[cohort, 'attendance_rate']
    print(f"  • Khóa {cohort}: {students} sinh viên - Tỷ lệ điểm danh {rate}%")

print("PHÂN TÍCH THEO THỜI GIAN")

print("PHÂN TÍCH THEO THÁNG:")
monthly_summary = df_final.groupby(['year', 'month']).agg({
    'student_id': 'count',
}).rename(columns={'student_id': 'total_records'})

monthly_status = df_final.groupby(['year', 'month', 'status']).size().unstack(fill_value=0)
monthly_summary = monthly_summary.join(monthly_status)

if 'Có mặt' in monthly_summary.columns:
    monthly_summary['attendance_rate'] = (
        monthly_summary['Có mặt'] / monthly_summary['total_records'] * 100
    ).round(2)
else:
    monthly_summary['attendance_rate'] = 0

print(monthly_summary.to_string())

print("PHÂN TÍCH THEO THỨ TRONG TUẦN:")
weekday_summary = df_final.groupby('day_of_week').agg({
    'student_id': 'count',
}).rename(columns={'student_id': 'total_records'})

weekday_status = df_final.groupby(['day_of_week', 'status']).size().unstack(fill_value=0)
weekday_summary = weekday_summary.join(weekday_status)

if 'Có mặt' in weekday_summary.columns:
    weekday_summary['attendance_rate'] = (
        weekday_summary['Có mặt'] / weekday_summary['total_records'] * 100
    ).round(2)
else:
    weekday_summary['attendance_rate'] = 0

print(weekday_summary.to_string())

print("PHÂN TÍCH THEO TỪNG NGÀY:")
daily_summary = df_final.groupby('date').agg({
    'student_id': 'count',
    'class_id': 'nunique',
}).rename(columns={'student_id': 'total_records', 'class_id': 'num_classes'})

daily_status = df_final.groupby(['date', 'status']).size().unstack(fill_value=0)
daily_summary = daily_summary.join(daily_status)

if 'Có mặt' in daily_summary.columns:
    daily_summary['attendance_rate'] = (
        daily_summary['Có mặt'] / daily_summary['total_records'] * 100
    ).round(2)
else:
    daily_summary['attendance_rate'] = 0

print(daily_summary.to_string())